# 파라미터 튜닝 실험

제어기 및 유도 파라미터가 경로 추종 성능에 미치는 영향을 실험적으로 확인합니다.

This notebook explores three tuning experiments:
1. PID proportional gain (K_P) variation
2. VFG convergence gain (k_e) variation
3. Sensor noise on/off comparison

**Author:** Suwon Lee, Kookmin University

In [ ]:
# -*- coding: utf-8 -*-
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from vfg_pathfollowing import (
    Simulator, StepCurvaturePath, PIDFeedforward, compute_metrics
)

## 공통 경로 설정

모든 실험에서 동일한 step curvature 경로를 사용합니다.

In [ ]:
# 공통 경로: 직선 5m -> 원호(R=0.5, 90도) -> 직선 25m
path = StepCurvaturePath(R=0.5, theta_arc=np.pi/2, L2=25)
T_SIM = 20.0
SPEED = 1.5

## 실험 1: PID 비례 게인 (K_P) 변화

K_P를 1.0, 2.0, 3.0, 5.0으로 변화시키며 방위각 오차 응답을 비교합니다. K_P가 증가하면 응답이 빨라지지만, 과도하면 진동이 발생할 수 있습니다.

In [ ]:
kp_values = [1.0, 2.0, 3.0, 5.0]
colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for kp, c in zip(kp_values, colors):
    # PID 제어기 인스턴스 생성 (K_P만 변경)
    pid = PIDFeedforward(K_P=kp)
    sim = Simulator(path, controller=pid, speed=SPEED)
    result = sim.run(T=T_SIM)
    m = compute_metrics(result)
    
    label = f'K_P={kp:.1f} (RMS={m["rms_e_psi_deg"]:.2f} deg)'
    axes[0].plot(result.time, np.degrees(result.e_psi), color=c, linewidth=1.2, label=label)
    axes[1].plot(result.time, result.e_d, color=c, linewidth=1.2, label=f'K_P={kp:.1f}')

axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('e_psi [deg]')
axes[0].set_title('Heading Error: K_P Sweep')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('e_d [m]')
axes[1].set_title('Cross-track Error: K_P Sweep')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 실험 2: VFG 수렴 게인 (k_e) 변화

VFG의 수렴 게인 k_e를 변화시킵니다. k_e가 클수록 경로로 빠르게 수렴하지만, 곡률 구간에서 과도한 조향을 유발할 수 있습니다. LPV H-inf 제어기를 사용합니다.

In [ ]:
ke_values = [1.0, 3.0, 5.0, 10.0]
colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ke, c in zip(ke_values, colors):
    # k_e만 변경하여 시뮬레이터 생성
    sim = Simulator(path, controller='lpv-hinf', speed=SPEED, k_e=ke)
    result = sim.run(T=T_SIM)
    m = compute_metrics(result)
    
    label = f'k_e={ke:.1f} (RMS={m["rms_e_psi_deg"]:.2f} deg)'
    axes[0].plot(result.time, np.degrees(result.e_psi), color=c, linewidth=1.2, label=label)
    axes[1].plot(result.time, result.e_d, color=c, linewidth=1.2, label=f'k_e={ke:.1f}')

axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('e_psi [deg]')
axes[0].set_title('Heading Error: k_e Sweep (LPV H-inf)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('e_d [m]')
axes[1].set_title('Cross-track Error: k_e Sweep (LPV H-inf)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 실험 3: 센서 노이즈 On/Off 비교

센서 노이즈가 있는 경우와 없는 경우의 제어 성능을 비교합니다. LPV H-inf 제어기는 H-infinity 설계 특성상 노이즈에 대한 강건성이 높습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

noise_cases = [
    (False, 'No Noise', '#1f77b4', '-'),
    (True,  'With Noise', '#d62728', '-'),
]

for noise_on, label, c, ls in noise_cases:
    sim = Simulator(path, controller='lpv-hinf', speed=SPEED,
                    noise=noise_on, noise_seed=42)
    result = sim.run(T=T_SIM)
    m = compute_metrics(result)
    
    full_label = f'{label} (RMS={m["rms_e_psi_deg"]:.2f} deg)'
    axes[0].plot(result.time, np.degrees(result.e_psi),
                 color=c, linestyle=ls, linewidth=1.0, label=full_label, alpha=0.8)
    axes[1].plot(result.time, result.e_d,
                 color=c, linestyle=ls, linewidth=1.0, label=label, alpha=0.8)

axes[0].set_xlabel('Time [s]')
axes[0].set_ylabel('e_psi [deg]')
axes[0].set_title('Heading Error: Noise Comparison (LPV H-inf)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Time [s]')
axes[1].set_ylabel('e_d [m]')
axes[1].set_title('Cross-track Error: Noise Comparison (LPV H-inf)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()